In [5]:
import csv
import re
import unicodedata
from pathlib import Path
from collections import defaultdict, Counter
import pymupdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

KINDS = {"brief", "bulletin", "manual", "article", "guideline", "leaflet"}

# Point 2: Exact normalisation applied identically to phrases, pages, and chunks
def norm_text(text, join_hyphen_linebreaks=False):
    s = unicodedata.normalize("NFKC", text)
    if join_hyphen_linebreaks:
        # Remove hyphen + line-break joins (e.g. "herbi-\ncides" -> "herbicides")
        s = re.sub(r"[-‐‑‒–—−]\s*\n\s*", "", s)
    s = s.lower()
    # Map en dash, em dash, and minus sign to ASCII '-'
    s = s.replace("\u2013", "-").replace("\u2014", "-").replace("\u2212", "-")
    # Remove whitespace before '%'
    s = re.sub(r"\s+%", "%", s)
    # Collapse all whitespace
    return " ".join(s.split())

def meta_from_name(path):
    crop, topic, org, year, kind = path.stem.split("_")
    assert kind in KINDS, f"unexpected kind {kind!r} in {path.name}"
    return {"doc_id": path.stem, "crop": crop, "topic": topic, "org": org, "year": int(year), "kind": kind}

def load_and_clean_corpus(raw_dir="../data/raw"):
    pages = []
    for p in sorted(Path(raw_dir).glob("*.pdf")):
        with pymupdf.open(str(p)) as pdf:
            for i, page in enumerate(pdf):
                meta = meta_from_name(path=p)
                meta.update({"source": p.name, "page": i + 1})
                pages.append(Document(page_content=page.get_text(), metadata=meta))

    by_doc = defaultdict(list)
    for d in pages:
        by_doc[d.metadata["doc_id"]].append(d)

    for doc_id, doc_pages in by_doc.items():
        counts = Counter(
            l.strip() for d in doc_pages
            for l in set(d.page_content.splitlines()) if l.strip()
        )
        cutoff = max(5, 0.5 * len(doc_pages))
        boiler = {l for l, n in counts.items() if n >= cutoff and len(l) >= 10}
        for d in doc_pages:
            text = d.page_content.replace("\uf0b7", "•")
            d.page_content = "\n".join(l for l in text.splitlines() if l.strip() not in boiler)
    return pages

def build_chunks_with_ids(pages_list, chunk_size=1000, chunk_overlap=100, separators=None, min_len=0):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators or ["\n\n", "\n", " ", ""],
    )
    raw_chunks = splitter.split_documents(pages_list)
    seen = Counter()
    chunks_out = []
    for c in raw_chunks:
        if len(c.page_content.strip()) < min_len:
            continue
        key = (c.metadata["doc_id"], int(c.metadata["page"]))
        cid = f"{key[0]}:p{key[1]}:c{seen[key]}"
        seen[key] += 1
        c.metadata = dict(c.metadata)
        c.metadata["chunk_id"] = cid
        chunks_out.append(c)
    return chunks_out

pages = load_and_clean_corpus("../data/raw")
baseline_chunks = build_chunks_with_ids(pages, chunk_size=1000, chunk_overlap=100)

# Point 3: Linter checks (a), (b), (c), (d) + write gold_chunk_ids column
def lint_and_populate_gold_chunks(csv_path="../eval/questions.csv", join_hyphens=False):
    page_map = {(d.metadata["doc_id"], int(d.metadata["page"])): norm_text(d.page_content, join_hyphens) for d in pages}
    
    # Group chunks by (doc_id, page)
    chunks_by_page = defaultdict(list)
    for c in baseline_chunks:
        key = (c.metadata["doc_id"], int(c.metadata["page"]))
        chunks_by_page[key].append((c.metadata["chunk_id"], norm_text(c.page_content, join_hyphens)))

    with open(csv_path, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames)
        rows = list(reader)

    if "gold_chunk_ids" not in fieldnames:
        fieldnames.append("gold_chunk_ids")

    errors = []
    seen_ids = set()
    by_split = Counter()
    by_crop = Counter()
    by_tag = Counter()
    hyphen_fixed_only = []

    for row_num, r in enumerate(rows, start=2):
        qid = r["id"].strip()
        if qid in seen_ids:
            errors.append(f"Row {row_num} ({qid}): duplicate id")
        seen_ids.add(qid)

        qtype = r["type"].strip()
        split_val = r.get("split", "").strip()
        tag = r.get("tag", "").strip()
        doc_id = r.get("doc_id", "").strip()
        crop = doc_id.split("_")[0] if doc_id else "out_of_scope"

        by_split[(split_val, qtype)] += 1
        by_crop[crop] += 1
        by_tag[tag] += 1

        if split_val not in {"dev", "test"}:
            errors.append(f"Row {row_num} ({qid}): invalid split {split_val!r}")

        pages_raw = [p.strip() for p in r.get("expected_pages", "").split(";") if p.strip()]
        phrases_raw = [norm_text(p, join_hyphens) for p in r.get("answer_phrases", "").split(";") if p.strip()]
        match_rule = r.get("match", "").strip()
        notes = r.get("notes", "").strip()

        if qtype == "in_scope":
            if not doc_id or not pages_raw or not phrases_raw or match_rule not in {"any", "all"}:
                errors.append(f"Row {row_num} ({qid}): incomplete in_scope fields")
                continue

            exp_pages = [int(p) for p in pages_raw]
            for pnum in exp_pages:
                if (doc_id, pnum) not in page_map:
                    errors.append(f"Row {row_num} ({qid}): page ({doc_id}, p.{pnum}) does not exist")

            matched_cids = []
            for ph in phrases_raw:
                # (a) Check phrase in expected page text
                page_hits = [pnum for pnum in exp_pages if ph in page_map.get((doc_id, pnum), "")]
                if not page_hits:
                    # Check if hyphen-join variant would fix it
                    alt_ph = norm_text(ph, join_hyphen_linebreaks=True)
                    alt_hits = [
                        pnum for pnum in exp_pages
                        if alt_ph in norm_text(
                            next((d.page_content for d in pages if d.metadata["doc_id"] == doc_id and d.metadata["page"] == pnum), ""),
                            join_hyphen_linebreaks=True
                        )
                    ]
                    if alt_hits:
                        hyphen_fixed_only.append((qid, ph, alt_hits))
                    errors.append(f"Row {row_num} ({qid}): phrase {ph!r} NOT found on expected pages {exp_pages} of {doc_id}")
                    continue

                # (b) Check phrase inside at least one indexed chunk on expected pages
                ph_cids = [
                    cid for pnum in exp_pages
                    for cid, ctext in chunks_by_page.get((doc_id, pnum), [])
                    if ph in ctext
                ]
                if not ph_cids:
                    errors.append(
                        f"Row {row_num} ({qid}): phrase {ph!r} is on page(s) {page_hits} of {doc_id}, "
                        f"but NOT in any indexed chunk (cut at chunk boundary or flattened table)!"
                    )
                else:
                    for cid in ph_cids:
                        if cid not in matched_cids:
                            matched_cids.append(cid)

            r["gold_chunk_ids"] = ";".join(matched_cids)

        elif qtype == "out_of_scope":
            r["gold_chunk_ids"] = ""
            if doc_id or pages_raw or phrases_raw:
                errors.append(f"Row {row_num} ({qid}): out_of_scope row must have empty doc_id, expected_pages, answer_phrases")
            # (c) Word-boundary check for every term after 'absent:' in notes
            if "absent:" in notes.lower():
                after = notes.lower().split("absent:", 1)[1]
                terms = [norm_text(t, join_hyphens) for t in after.split(";") if t.strip()]
                for kt in terms:
                    pattern = re.compile(r"\b" + re.escape(kt) + r"\b")
                    found_pages = [
                        f"{doc}:p{pnum}" for (doc, pnum), ptext in page_map.items()
                        if pattern.search(ptext)
                    ]
                    if found_pages:
                        errors.append(f"Row {row_num} ({qid}): absent term {kt!r} found on {found_pages[:3]}")
            else:
                errors.append(f"Row {row_num} ({qid}): out_of_scope notes must include 'absent: term1; term2'")
        else:
            errors.append(f"Row {row_num} ({qid}): invalid type {qtype!r}")

    # (d) Print counts by split, crop, and tag
    print("=== LINTER SUMMARY (join_hyphen_linebreaks=" + str(join_hyphens) + ") ===")
    print("Counts by (split, type):", dict(sorted(by_split.items())))
    print("Counts by crop:         ", dict(sorted(by_crop.items())))
    print("Counts by tag:          ", dict(sorted(by_tag.items())))
    if hyphen_fixed_only:
        print("Phrases only found when hyphen+linebreak joins are removed:", hyphen_fixed_only)

    if errors:
        print(f"\nLINTER FAILED WITH {len(errors)} ERROR(S):")
        for e in errors:
            print("  [ERROR]", e)
    else:
        with open(csv_path, mode="w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        print(f"\nLINTER PASSED (0 errors)! Updated {csv_path} with 'gold_chunk_ids' column.")
    return errors

# Run standard normalisation and also test the hyphen+linebreak variant (Point 2)
err_std = lint_and_populate_gold_chunks("../eval/questions.csv", join_hyphens=False)
print("-" * 70)
err_hyp = lint_and_populate_gold_chunks("../eval/questions.csv", join_hyphens=True)

=== LINTER SUMMARY (join_hyphen_linebreaks=False) ===
Counts by (split, type): {('dev', 'in_scope'): 20, ('dev', 'out_of_scope'): 6, ('test', 'in_scope'): 10, ('test', 'out_of_scope'): 4}
Counts by crop:          {'out_of_scope': 10, 'potato': 16, 'rice': 10, 'wheat': 4}
Counts by tag:           {'conflict': 1, 'multi': 6, 'out_of_scope': 10, 'paraphrase': 8, 'plain': 8, 'table': 7}

LINTER PASSED (0 errors)! Updated ../eval/questions.csv with 'gold_chunk_ids' column.
----------------------------------------------------------------------
=== LINTER SUMMARY (join_hyphen_linebreaks=True) ===
Counts by (split, type): {('dev', 'in_scope'): 20, ('dev', 'out_of_scope'): 6, ('test', 'in_scope'): 10, ('test', 'out_of_scope'): 4}
Counts by crop:          {'out_of_scope': 10, 'potato': 16, 'rice': 10, 'wheat': 4}
Counts by tag:           {'conflict': 1, 'multi': 6, 'out_of_scope': 10, 'paraphrase': 8, 'plain': 8, 'table': 7}

LINTER PASSED (0 errors)! Updated ../eval/questions.csv with 'gold_chu

In [6]:
import sqlite3
import hashlib
import json
import time
from datetime import datetime
import numpy as np
from langchain_ollama import OllamaEmbeddings

class CachedEmbedder:
    def __init__(self, model_name="nomic-embed-text", doc_prefix="", query_prefix="", cache_path="../data/embed_cache.sqlite"):
        self.model_name = model_name
        self.doc_prefix = doc_prefix
        self.query_prefix = query_prefix
        self.emb = OllamaEmbeddings(model=model_name)
        self.conn = sqlite3.connect(cache_path)
        self.conn.execute("CREATE TABLE IF NOT EXISTS embeddings (cache_key TEXT PRIMARY KEY, vec_json TEXT NOT NULL)")
        self.conn.commit()

    def _key(self, prefix, text):
        return hashlib.sha256(f"{self.model_name}\0{prefix}\0{text}".encode("utf-8")).hexdigest()

    def embed_texts(self, texts, is_query=False, batch_size=32):
        prefix = self.query_prefix if is_query else self.doc_prefix
        keys = [self._key(prefix, t) for t in texts]
        results = [None] * len(texts)
        missing = []
        cur = self.conn.cursor()
        for idx, k in enumerate(keys):
            row = cur.execute("SELECT vec_json FROM embeddings WHERE cache_key = ?", (k,)).fetchone()
            if row is not None:
                results[idx] = json.loads(row[0])
            else:
                missing.append(idx)
        if missing:
            for s in range(0, len(missing), batch_size):
                b_idx = missing[s:s + batch_size]
                b_in = [prefix + texts[i] for i in b_idx]
                vecs = [self.emb.embed_query(b_in[0])] if (is_query and len(b_in) == 1) else self.emb.embed_documents(b_in)
                for i, v in zip(b_idx, vecs):
                    results[i] = v
                    cur.execute("INSERT OR REPLACE INTO embeddings (cache_key, vec_json) VALUES (?, ?)", (keys[i], json.dumps(v)))
                self.conn.commit()
        mat = np.array(results, dtype=np.float64)
        norms = np.linalg.norm(mat, axis=1, keepdims=True)
        return mat / np.clip(norms, 1e-12, None)

def load_eval_questions(csv_file="../eval/questions.csv", split_filter="dev", join_hyphens=False):
    rows = []
    with open(csv_file, mode="r", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            if split_filter and r["split"].strip() != split_filter:
                continue
            rows.append({
                "id": r["id"].strip(),
                "question": r["question"].strip(),
                "type": r["type"].strip(),
                "tag": r["tag"].strip(),
                "doc_id": r["doc_id"].strip(),
                "expected_pages": [int(p.strip()) for p in r["expected_pages"].split(";") if p.strip()],
                "answer_phrases": [norm_text(p, join_hyphens) for p in r["answer_phrases"].split(";") if p.strip()],
                "match": r["match"].strip(),
                "split": r["split"].strip(),
            })
    return rows

# Point 4: Hit rule — page in expected_pages and phrase in chunk text (union of top-k for match='all')
def chunk_matches_phrase(c_meta, c_text_norm, q, ph):
    return (c_meta["doc_id"] == q["doc_id"]) and (int(c_meta["page"]) in q["expected_pages"]) and (ph in c_text_norm)

def is_hit_at_k(ranked_chunks, q, k):
    top_k = ranked_chunks[:k]
    if q["match"] == "any":
        return any(chunk_matches_phrase(c["meta"], c["text_norm"], q, ph) for c in top_k for ph in q["answer_phrases"])
    elif q["match"] == "all":
        return all(any(chunk_matches_phrase(c["meta"], c["text_norm"], q, ph) for c in top_k) for ph in q["answer_phrases"])
    return False

EXP_CSV = Path("../eval/results/experiments.csv")
EXP_CSV.parent.mkdir(parents=True, exist_ok=True)
if not EXP_CSV.exists():
    with open(EXP_CSV, mode="w", encoding="utf-8", newline="") as f:
        csv.writer(f).writerow([
            "run_id", "date", "config", "split", "n", "recall@1", "recall@3", "recall@5",
            "mrr@10", "median_gold_rank", "oos_false_accept_rate", "is_false_reject_rate", "notes"
        ])

def summarize_group(records):
    if not records:
        return {"n": 0, "recall@1": 0.0, "recall@3": 0.0, "recall@5": 0.0, "mrr@10": 0.0, "median_gold_rank": 0.0}
    return {
        "n": len(records),
        "recall@1": round(float(np.mean([x["r1"] for x in records])), 4),
        "recall@3": round(float(np.mean([x["r3"] for x in records])), 4),
        "recall@5": round(float(np.mean([x["rk"] for x in records])), 4),
        "mrr@10": round(float(np.mean([x["rr10"] for x in records])), 4),
        "median_gold_rank": float(np.median([x["gold_rank"] for x in records])),
    }

def run_config(config, pages_input=None, split="dev", threshold=0.765, notes="", verbose_tags=True):
    if pages_input is None:
        pages_input = pages
    join_hyp = config.get("join_hyphens", False)
    chunks = build_chunks_with_ids(
        pages_input,
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        separators=config.get("separators", ["\n\n", "\n", " ", ""]),
        min_len=config.get("min_chunk_len", 0),
    )

    embedder = CachedEmbedder(
        model_name=config.get("embed_model", "nomic-embed-text"),
        doc_prefix=config.get("doc_prefix", ""),
        query_prefix=config.get("query_prefix", ""),
    )
    chunk_texts = [c.page_content for c in chunks]
    C_mat = embedder.embed_texts(chunk_texts, is_query=False)
    chunk_norms = [norm_text(t, join_hyp) for t in chunk_texts]

    qs = load_eval_questions(split_filter=split, join_hyphens=join_hyp)
    q_vecs = embedder.embed_texts([q["question"] for q in qs], is_query=True)
    sim_matrix = q_vecs @ C_mat.T

    k_eval = config.get("k", 5)
    in_records = []
    by_tag_records = defaultdict(list)
    in_top1_sims, out_top1_sims = [], []

    for qi, q in enumerate(qs):
        sims = sim_matrix[qi]
        order = np.argsort(-sims)
        top1_sim = float(sims[order[0]])
        ranked = [{"meta": chunks[idx].metadata, "text_norm": chunk_norms[idx], "sim": float(sims[idx])} for idx in order]

        if q["type"] == "in_scope":
            in_top1_sims.append((q["id"], q["tag"], top1_sim))
            hit_rank = None
            for rank_i in range(1, len(ranked) + 1):
                if is_hit_at_k(ranked, q, rank_i):
                    hit_rank = rank_i
                    break
            rec = {
                "id": q["id"],
                "tag": q["tag"],
                "r1": 1.0 if is_hit_at_k(ranked, q, 1) else 0.0,
                "r3": 1.0 if is_hit_at_k(ranked, q, 3) else 0.0,
                "rk": 1.0 if is_hit_at_k(ranked, q, k_eval) else 0.0,
                "rr10": (1.0 / hit_rank) if (hit_rank is not None and hit_rank <= 10) else 0.0,
                "gold_rank": hit_rank if hit_rank is not None else len(ranked),
                "top1_sim": top1_sim,
            }
            in_records.append(rec)
            by_tag_records[q["tag"]].append(rec)
        else:
            out_top1_sims.append((q["id"], q["tag"], top1_sim))

    overall = summarize_group(in_records)
    per_tag = {t: summarize_group(recs) for t, recs in sorted(by_tag_records.items())}
    table_stats = per_tag.get("table", {"n": 0, "recall@5": 0.0, "mrr@10": 0.0})

    oos_fa = sum(1 for _, _, s in out_top1_sims if s >= threshold) / max(1, len(out_top1_sims))
    is_fr = sum(1 for _, _, s in in_top1_sims if s < threshold) / max(1, len(in_top1_sims))

    full_notes = f"chunks={len(chunks)}; table_R@{k_eval}={table_stats['recall@5']:.3f}; {notes}"
    metrics = {
        "run_id": config["run_id"],
        "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "config": json.dumps({k: v for k, v in config.items() if k != "run_id"}),
        "split": split,
        "n": len(qs),
        "recall@1": overall["recall@1"],
        "recall@3": overall["recall@3"],
        "recall@5": overall["recall@5"],
        "mrr@10": overall["mrr@10"],
        "median_gold_rank": overall["median_gold_rank"],
        "oos_false_accept_rate": round(oos_fa, 4),
        "is_false_reject_rate": round(is_fr, 4),
        "notes": full_notes,
        "per_tag": per_tag,
        "in_records": in_records,
        "in_top1_sims": in_top1_sims,
        "out_top1_sims": out_top1_sims,
    }

    with open(EXP_CSV, mode="a", encoding="utf-8", newline="") as f:
        csv.writer(f).writerow([
            metrics["run_id"], metrics["date"], metrics["config"], metrics["split"], metrics["n"],
            metrics["recall@1"], metrics["recall@3"], metrics["recall@5"], metrics["mrr@10"],
            metrics["median_gold_rank"], metrics["oos_false_accept_rate"], metrics["is_false_reject_rate"], metrics["notes"]
        ])

    if verbose_tags:
        print(f"\n[{metrics['run_id']}] (split={split}, n={len(qs)}, chunks={len(chunks)})")
        print(f"  OVERALL (n={overall['n']:<2}): R@1={overall['recall@1']:.3f} | R@3={overall['recall@3']:.3f} | R@{k_eval}={overall['recall@5']:.3f} | MRR@10={overall['mrr@10']:.3f} | MedRank={overall['median_gold_rank']}")
        for t, st in per_tag.items():
            print(f"  tag={t:<10} (n={st['n']:<2}): R@1={st['recall@1']:.3f} | R@3={st['recall@3']:.3f} | R@{k_eval}={st['recall@5']:.3f} | MRR@10={st['mrr@10']:.3f} | MedRank={st['median_gold_rank']}")
    return metrics

# 5A-3: Run Baseline (1000/100, nomic-embed-text, no prefix, k=5) on dev split
base_res = run_config(
    {"run_id": "5A3_baseline_1000_100_k5", "chunk_size": 1000, "chunk_overlap": 100, "k": 5},
    notes="Stage 5A-3 baseline"
)
print("\nPer-question ranks on dev baseline:")
for rec in base_res["in_records"]:
    print(f"  {rec['id']:<3} ({rec['tag']:<10}): gold_rank={rec['gold_rank']:<3} | top1_sim={rec['top1_sim']:.4f}")


[5A3_baseline_1000_100_k5] (split=dev, n=26, chunks=615)
  OVERALL (n=20): R@1=0.400 | R@3=0.700 | R@5=0.750 | MRR@10=0.565 | MedRank=2.0
  tag=conflict   (n=1 ): R@1=0.000 | R@3=0.000 | R@5=0.000 | MRR@10=0.100 | MedRank=10.0
  tag=multi      (n=4 ): R@1=0.250 | R@3=0.500 | R@5=0.750 | MRR@10=0.438 | MedRank=3.0
  tag=paraphrase (n=5 ): R@1=0.200 | R@3=0.800 | R@5=0.800 | MRR@10=0.467 | MedRank=2.0
  tag=plain      (n=7 ): R@1=0.714 | R@3=0.857 | R@5=0.857 | MRR@10=0.786 | MedRank=1.0
  tag=table      (n=3 ): R@1=0.333 | R@3=0.667 | R@5=0.667 | MRR@10=0.537 | MedRank=2.0

Per-question ranks on dev baseline:
  q1  (table     ): gold_rank=9   | top1_sim=0.8901
  q2  (multi     ): gold_rank=1   | top1_sim=0.8391
  q3  (plain     ): gold_rank=1   | top1_sim=0.8141
  q4  (conflict  ): gold_rank=10  | top1_sim=0.7722
  q5  (plain     ): gold_rank=3   | top1_sim=0.8067
  q6  (plain     ): gold_rank=1   | top1_sim=0.8585
  q7  (table     ): gold_rank=2   | top1_sim=0.7776
  q8  (paraphrase):

In [7]:
# 5A-4: Chunk size {500, 1000, 1500} x overlap {0, 100, 200}
grid_results = []
for c_size in [500, 1000, 1500]:
    for c_ov in [0, 100, 200]:
        res = run_config(
            {"run_id": f"5A4_grid_{c_size}_{c_ov}_k5", "chunk_size": c_size, "chunk_overlap": c_ov, "k": 5},
            notes=f"grid {c_size}/{c_ov}",
            verbose_tags=False,
        )
        grid_results.append(res)
        tbl_r5 = res["per_tag"]["table"]["recall@5"]
        print(f"size={c_size:<4} ov={c_ov:<3} | R@1={res['recall@1']:.3f} R@3={res['recall@3']:.3f} R@5={res['recall@5']:.3f} | table_R@5={tbl_r5:.3f} | MRR@10={res['mrr@10']:.3f} | MedRank={res['median_gold_rank']}")

# Check k in {3, 4, 6} on 1000/100
for k_val in [3, 4, 6]:
    res_k = run_config(
        {"run_id": f"5A4_k_{k_val}_1000_100", "chunk_size": 1000, "chunk_overlap": 100, "k": k_val},
        notes=f"k={k_val} check",
        verbose_tags=False,
    )
    print(f"k={k_val} (1000/100) -> Recall@{k_val}={res_k['recall@5']:.3f} | table_R@{k_val}={res_k['per_tag']['table']['recall@5']:.3f} | MRR@10={res_k['mrr@10']:.3f}")

# 5A-6: Threshold analysis on dev split (target: in-scope false-reject <= 10%)
in_sims = sorted([s for _, _, s in base_res["in_top1_sims"]])
out_sims = sorted([s for _, _, s in base_res["out_top1_sims"]])

print("\n=== 5A-6 DEV SPLIT TOP-1 SIMILARITIES ===")
for qid, tag, s in sorted(base_res["in_top1_sims"], key=lambda x: x[2]):
    print(f"  IN_SCOPE     {qid:<3} ({tag:<10}): {s:.4f}")
for qid, tag, s in sorted(base_res["out_top1_sims"], key=lambda x: -x[2]):
    print(f"  OUT_OF_SCOPE {qid:<3} ({tag:<10}): {s:.4f}")

print(f"\nMin dev in-scope top-1:     {min(in_sims):.4f}")
print(f"Max dev out-of-scope top-1: {max(out_sims):.4f}")

candidate_taus = sorted(set([round(s, 4) for s in in_sims + out_sims]))
best_tau = None
for tau in candidate_taus:
    fr = sum(1 for s in in_sims if s < tau) / len(in_sims)
    fa = sum(1 for s in out_sims if s >= tau) / len(out_sims)
    if fr <= 0.10:
        if best_tau is None or fa < best_tau[2] or (fa == best_tau[2] and tau > best_tau[0]):
            best_tau = (tau, fr, fa)

print(f"Chosen Dev Threshold (FR <= 10%): tau={best_tau[0]:.4f} | In-Scope FR={best_tau[1]*100:.1f}% | Out-of-Scope FA={best_tau[2]*100:.1f}%")
if min(in_sims) <= max(out_sims):
    print("NOTE: Dev in-scope and out-of-scope distributions overlap! A second lexical/crop-entity signal is recommended.")

size=500  ov=0   | R@1=0.550 R@3=0.800 R@5=0.900 | table_R@5=1.000 | MRR@10=0.678 | MedRank=1.0
size=500  ov=100 | R@1=0.400 R@3=0.700 R@5=0.850 | table_R@5=1.000 | MRR@10=0.583 | MedRank=2.0
size=500  ov=200 | R@1=0.500 R@3=0.700 R@5=0.800 | table_R@5=1.000 | MRR@10=0.644 | MedRank=1.5
size=1000 ov=0   | R@1=0.500 R@3=0.750 R@5=0.800 | table_R@5=0.667 | MRR@10=0.610 | MedRank=1.5
size=1000 ov=100 | R@1=0.400 R@3=0.700 R@5=0.750 | table_R@5=0.667 | MRR@10=0.565 | MedRank=2.0
size=1000 ov=200 | R@1=0.400 R@3=0.750 R@5=0.850 | table_R@5=0.667 | MRR@10=0.581 | MedRank=2.0
size=1500 ov=0   | R@1=0.350 R@3=0.450 R@5=0.600 | table_R@5=0.667 | MRR@10=0.468 | MedRank=5.0
size=1500 ov=100 | R@1=0.300 R@3=0.500 R@5=0.650 | table_R@5=0.667 | MRR@10=0.458 | MedRank=3.5
size=1500 ov=200 | R@1=0.350 R@3=0.450 R@5=0.750 | table_R@5=1.000 | MRR@10=0.482 | MedRank=4.0
k=3 (1000/100) -> Recall@3=0.700 | table_R@3=0.667 | MRR@10=0.565
k=4 (1000/100) -> Recall@4=0.750 | table_R@4=0.667 | MRR@10=0.565
k=6 

In [8]:
# 5B-1: Compare nomic with task prefixes & bge-m3 on (1000, 100)
res_nomic_pref = run_config({
    "run_id": "5B1_nomic_prefixes", "chunk_size": 1000, "chunk_overlap": 100, "k": 5,
    "embed_model": "nomic-embed-text", "doc_prefix": "search_document: ", "query_prefix": "search_query: "
}, notes="nomic with search_document/search_query prefixes")

# Check if bge-m3 vectors are unit length first
bge_test = CachedEmbedder(model_name="bge-m3").embed_texts(["test unit length"], is_query=True)
print("bge-m3 norm check:", np.linalg.norm(bge_test, axis=1))

t0 = time.perf_counter()
res_bge = run_config({
    "run_id": "5B1_bge_m3", "chunk_size": 1000, "chunk_overlap": 100, "k": 5,
    "embed_model": "bge-m3", "doc_prefix": "", "query_prefix": ""
}, notes="bge-m3 1024-dim")
print(f"bge-m3 run completed in {round(time.perf_counter() - t0, 1)}s")

# 5B-2: Table-aware extraction (extract tables via to_markdown() and redact bbox from page to avoid duplicating raw flattened text)
def load_corpus_table_aware(raw_dir="../data/raw"):
    prose_pages = []
    table_chunks = []

    for p in sorted(Path(raw_dir).glob("*.pdf")):
        with pymupdf.open(str(p)) as pdf:
            for i, page in enumerate(pdf):
                meta = meta_from_name(p)
                meta.update({"source": p.name, "page": i + 1})
                full_lines = [l.strip() for l in page.get_text().splitlines() if l.strip()]

                tabs = page.find_tables()
                if tabs.tables:
                    for ti, tb in enumerate(tabs.tables):
                        md_str = tb.to_markdown()
                        if len(md_str.strip()) >= 40:
                            # Grab up to 3 lines above the table as caption context
                            caption = " ".join(full_lines[:3])
                            t_meta = dict(meta)
                            t_meta.update({"is_table": True, "table_idx": ti})
                            table_chunks.append(Document(page_content=f"{caption}\n{md_str}", metadata=t_meta))
                        # Redact table bbox on in-memory page so raw flattened cells are not duplicated
                        page.add_redact_annot(tb.bbox)
                    page.apply_redactions()

                prose_pages.append(Document(page_content=page.get_text(), metadata=meta))

    # Apply same boilerplate header cleaning to prose_pages
    by_doc = defaultdict(list)
    for d in prose_pages:
        by_doc[d.metadata["doc_id"]].append(d)
    for doc_id, doc_pages in by_doc.items():
        counts = Counter(l.strip() for d in doc_pages for l in set(d.page_content.splitlines()) if l.strip())
        cutoff = max(5, 0.5 * len(doc_pages))
        boiler = {l for l, n in counts.items() if n >= cutoff and len(l) >= 10}
        for d in doc_pages:
            text = d.page_content.replace("\uf0b7", "•")
            d.page_content = "\n".join(l for l in text.splitlines() if l.strip() not in boiler)

    return prose_pages + table_chunks

pages_table_aware = load_corpus_table_aware("../data/raw")
res_tables = run_config(
    {"run_id": "5B2_table_aware_dedup", "chunk_size": 1000, "chunk_overlap": 100, "k": 5},
    pages_input=pages_table_aware,
    notes="find_tables().to_markdown() + bbox redacted"
)

# 5B-3: Test (a) reflowing wrapped lines + '. ' split point, and (b) dropping reference-list chunks & chunks < 50 chars separately
pages_reflowed = []
for d in pages:
    # Reflow single line breaks inside paragraphs and join hyphenated line wraps
    txt = re.sub(r"[-‐‑‒–—−]\s*\n\s*", "", d.page_content)
    txt = re.sub(r"(?<![.!?:])\n(?=[a-z0-9(])", " ", txt)
    pages_reflowed.append(Document(page_content=txt, metadata=dict(d.metadata)))

res_reflow = run_config(
    {
        "run_id": "5B3a_reflow_sentences", "chunk_size": 1000, "chunk_overlap": 100, "k": 5,
        "separators": ["\n\n", "\n", ". ", " ", ""], "join_hyphens": True
    },
    pages_input=pages_reflowed,
    notes="reflow wrapped lines + '. ' separator"
)

def is_reference_page_or_chunk(text):
    years = re.findall(r"\b(19\d\d|20\d\d)\b", text)
    return len(years) >= 6 and ("journal" in text.lower() or "pp." in text.lower() or "et al" in text.lower() or " Bull" in text)

pages_no_refs = [
    Document(page_content=d.page_content, metadata=dict(d.metadata))
    for d in pages if not is_reference_page_or_chunk(d.page_content)
]

res_no_refs = run_config(
    {
        "run_id": "5B3b_drop_refs_min50", "chunk_size": 1000, "chunk_overlap": 100, "k": 5,
        "min_chunk_len": 50
    },
    pages_input=pages_no_refs,
    notes="drop reference pages + min_chunk_len=50"
)


[5B1_nomic_prefixes] (split=dev, n=26, chunks=615)
  OVERALL (n=20): R@1=0.450 | R@3=0.600 | R@5=0.800 | MRR@10=0.581 | MedRank=2.0
  tag=conflict   (n=1 ): R@1=0.000 | R@3=1.000 | R@5=1.000 | MRR@10=0.333 | MedRank=3.0
  tag=multi      (n=4 ): R@1=0.250 | R@3=0.500 | R@5=0.750 | MRR@10=0.438 | MedRank=3.0
  tag=paraphrase (n=5 ): R@1=0.400 | R@3=0.600 | R@5=0.600 | MRR@10=0.533 | MedRank=2.0
  tag=plain      (n=7 ): R@1=0.714 | R@3=0.714 | R@5=1.000 | MRR@10=0.779 | MedRank=1.0
  tag=table      (n=3 ): R@1=0.333 | R@3=0.333 | R@5=0.667 | MRR@10=0.472 | MedRank=4.0
bge-m3 norm check: [1.]

[5B1_bge_m3] (split=dev, n=26, chunks=615)
  OVERALL (n=20): R@1=0.450 | R@3=0.750 | R@5=0.900 | MRR@10=0.637 | MedRank=2.0
  tag=conflict   (n=1 ): R@1=0.000 | R@3=0.000 | R@5=0.000 | MRR@10=0.167 | MedRank=6.0
  tag=multi      (n=4 ): R@1=0.250 | R@3=0.750 | R@5=1.000 | MRR@10=0.521 | MedRank=2.5
  tag=paraphrase (n=5 ): R@1=0.400 | R@3=0.600 | R@5=0.800 | MRR@10=0.550 | MedRank=2.0
  tag=plain   